# PolyglotSound sur Google Colab

Faites chanter une chanson dans une autre langue (fon, francais, anglais), sur sa melodie, avec la voix autorisee de l'artiste.

**Avant de commencer** : menu *Execution > Modifier le type d'execution > GPU T4*. Puis lancez les cellules dans l'ordre.

| Cellule | Duree (T4) | Role |
|---|---|---|
| 1 | 10 s | verifie le GPU et recupere le code |
| 2 | 10 a 15 min | installe tout (Python 3.11, PyTorch, modeles, Seed-VC, Ollama) et lance les tests |
| 3 | 3 a 6 min | demo complete sur un refrain fon **synthetique** (aucune voix reelle) |
| 4 | - | ouvre le studio artiste (lien public gradio.live) |

Licences : NLLB-200 et MMS-TTS sont **non commerciaux** (CC-BY-NC-4.0), Seed-VC est GPL-3.0. Usage R&D / demonstration. Ne convertissez jamais la voix d'une personne sans son accord ecrit.

In [ ]:
# 1. GPU + code
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'Pas de GPU : tout marchera sur CPU, en plus lent.'
import os
if not os.path.exists('/content/polyglotsound'):
    !git clone --depth 1 https://github.com/Souraka229/polyglotsound /content/polyglotsound
%cd /content/polyglotsound
!git log -1 --format='version : %h %cd'

In [ ]:
# 2. Installation complete + tests (WITH_LLM=0 pour sauter le LLM local)
%cd /content/polyglotsound
!WITH_LLM=1 bash scripts/setup_colab.sh 2>&1 | grep -v -E 'Downloading|Downloaded|Prepared|Resolved|Audited|^ \+ ' 

In [ ]:
# 3. Demo de bout en bout sur un refrain fon SYNTHETIQUE -> francais (mode Auto experimental)
%cd /content/polyglotsound
!.venv/bin/python scripts/make_synthetic_song.py
!.venv/bin/python scripts/demo_e2e.py fra --use-llm 2>&1 | grep -v -i warn
import json
from IPython.display import Audio, display
res = json.load(open('benchmarks/results/demo_e2e_fon_fra.json'))
print('Adaptation retenue :', res['chosen'])
print('Filigrane dans le mix :', res['watermark_check_mix'])
print('Original :'); display(Audio('datasets/synthetic/fon_refrain_01/mix.wav'))
print('Nouvelle version :'); display(Audio(res['wav']))

In [ ]:
# 4. Studio artiste : cliquez sur le lien https://xxxx.gradio.live qui s'affiche (la cellule reste active)
%cd /content/polyglotsound
!pgrep -x ollama >/dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)
!.venv/bin/python -m app.ui.gradio_app --share --host 0.0.0.0

### En cas de souci
- *Session redemarree / deconnectee* : relancez les cellules 1 puis 2 (les fichiers de `/content` sont effaces par Colab).
- *Pas de lien gradio.live* : relancez la cellule 4 ; le service de partage de Gradio peut etre momentanement indisponible.
- *Fon* : Whisper ne reconnait pas le fon, saisissez les paroles a l'etape 2 du studio.
- *Tests et benchmarks* : `!.venv/bin/python -m pytest -m models` puis `!.venv/bin/python benchmarks/bench_synthetic.py`.